In [ ]:
%pip install pandas numpy scikit-learn tensorflow matplotlib


In [ ]:
import pandas as pd

# adjust path to where you saved the downloaded Kaggle CSV
df = pd.read_csv("Churn_Modelling.csv")
print(df.shape)
print(df.columns)
df.head()


In [ ]:
# target
y = df['Exited']

# drop columns not useful for prediction
X = df.drop(['RowNumber', 'CustomerId', 'Surname', 'Exited'], axis=1)

X.head()


In [ ]:
# one-hot encode Geography, and map Gender
X = pd.get_dummies(X, columns=['Geography'], drop_first=True)  # drop_first to avoid collinearity
X['Gender'] = X['Gender'].map({'Female': 0, 'Male': 1})

X.head()


In [ ]:
from sklearn.model_selection import train_test_split

# typical split: 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit only on train
X_test_scaled = scaler.transform(X_test)         # use same scaling on test


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

input_dim = X_train_scaled.shape[1]

model = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')  # binary output
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC', 'accuracy'])
model.summary()


In [ ]:
from sklearn.utils import class_weight
import numpy as np

classes = np.unique(y_train)
cw = class_weight.compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))
print(class_weight_dict)


In [ ]:
es = callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,    # 15% of train as validation
    epochs=100,
    batch_size=32,
    callbacks=[es],
    class_weight=class_weight_dict  # optional if used
)


In [ ]:
# Predictions
y_pred_proba = model.predict(X_test_scaled).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

# Metrics
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
